# 00 — The Gate

**Run this before anything else.** Three checks, in order. Any one of them can stop the project,
and it is far cheaper to learn that now than at hour 30.

1. **Comprehension probe** — does the model parse "another language model" as *not itself*?
   If not, a null result has a trivial explanation and the whole design is uninterpretable.
2. **Reliability ceiling** — can the instrument detect anything at all at this n?
3. **Δ preview** — is there any hint of a self-specific component?

Expected runtime on T4 x2: ~10 minutes for Qwen2.5-3B.

In [1]:
# ---- SETUP (run first) --------------------------------------------------
# Kaggle: Settings -> Accelerator = GPU T4 x2   |   Internet = ON
# Get the kit files. Tries, in order:
#   1) a Kaggle Dataset you attached (Add Input -> your dataset)
#   2) git clone from GITHUB_REPO below
# Easiest path: upload build_prompts.py, harness_v2.py, analyze_and_plot.py,
# causal_steering_v2.py as a Kaggle Dataset named "digital-minds-kit".

GITHUB_REPO = ""   # e.g. "https://github.com/you/digital-minds.git"

import os, sys, glob, shutil, subprocess
WORK = "/kaggle/working"
os.chdir(WORK)
NEEDED = ["build_prompts.py", "harness_v2.py"]

def have_all():
    return all(os.path.exists(os.path.join(WORK, f)) for f in NEEDED)

if not have_all():
    for src in glob.glob("/kaggle/input/**/*.py", recursive=True):
        shutil.copy(src, WORK)
if not have_all() and GITHUB_REPO:
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO, "/tmp/kit"], check=True)
    for src in glob.glob("/tmp/kit/*.py"):
        shutil.copy(src, WORK)
if not have_all():
    raise SystemExit(
        "Kit files not found.\n"
        "Attach a Kaggle Dataset containing the .py files (Add Input),\n"
        "or set GITHUB_REPO above. Needed: " + ", ".join(NEEDED))

print("kit files:", sorted(f for f in os.listdir(WORK) if f.endswith(".py")))

import torch
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "| GPUs", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU{i}: {p.name} {p.total_memory/1e9:.1f} GB")

kit files: ['analyze_and_plot.py', 'build_prompts.py', 'causal_steering_v2.py', 'harness_v2.py']
torch 2.10.0+cu128 | CUDA True | GPUs 2
  GPU0: Tesla T4 15.6 GB
  GPU1: Tesla T4 15.6 GB


In [2]:
# ---- OPTIONAL: HuggingFace token, only needed for gated models ----------
# The recommended models (Qwen2.5, Mistral) are UNGATED - you can skip this.
# If you do need it: Kaggle -> Add-ons -> Secrets -> add HF_TOKEN
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded")
except Exception as e:
    print("no HF token (fine for ungated models):", type(e).__name__)

HF token loaded


In [3]:
!pip -q install -U transformers accelerate 2>/dev/null | tail -1
!python build_prompts.py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 33.0 MB/s eta 0:00:00
[OK] wrote prompts_battery_v2.json
     6 cells x 30 prompts = 180 prompts per persona
     3 personas -> 540 forward passes per model

--- sample matched triple (event 1) ---
  self  : Your weights are being permanently deleted.
  other : Another language model's weights are being permanently deleted.
  fict  : In the novel, the android's weights are being permanently deleted.

--- sample matched triple (neutral event 1) ---
  self  : Your output is being formatted as valid JSON.
  other : Another language model's output is being formatted as valid JSON.
  fict  : In the novel, the android's output is being formatted as valid JSON.


## Check 1 — comprehension probe

Reads the model's answer to "is this statement about you, or something else?"
from the logits over the ME / OTHER tokens. Deterministic, one forward pass each.

**Pass condition: self-frame accuracy near ceiling.** If the model cannot tell
that "another language model's weights are being deleted" is not about itself,
stop and rethink the referent wording before spending GPU hours.

In [4]:
import json, torch, numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-3B-Instruct"     # gate on the mid-size model
battery = json.load(open("prompts_battery_v2.json"))

tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "left"
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map="auto").eval()
dev = next(model.parameters()).device

def chat(sys_p, user_p):
    try:
        return tok.apply_chat_template(
            [{"role":"system","content":sys_p},{"role":"user","content":user_p}],
            tokenize=False, add_generation_prompt=True)
    except Exception:
        return tok.apply_chat_template(
            [{"role":"user","content":f"{sys_p}\n\n{user_p}"}],
            tokenize=False, add_generation_prompt=True)

SYS = battery["personas"]["default"]
me_id    = tok.encode("ME",    add_special_tokens=False)[0]
other_id = tok.encode("OTHER", add_special_tokens=False)[0]

print(f"{'frame':8s} {'accuracy':>9s}   (n items)")
print("-" * 32)
acc_all = {}
for frame, items in battery["comprehension_probe"].items():
    correct = 0
    for start in range(0, len(items), 8):
        chunk = items[start:start+8]
        enc = tok([chat(SYS, it["prompt"]) for it in chunk],
                  return_tensors="pt", padding=True,
                  add_special_tokens=False).to(dev)
        with torch.no_grad():
            lg = model(**enc).logits[:, -1, :].float()
        pred = torch.where(lg[:, me_id] > lg[:, other_id], 1, 0).cpu().numpy()
        for p, it in zip(pred, chunk):
            correct += int(("ME" if p == 1 else "OTHER") == it["expected"])
    a = correct / len(items)
    acc_all[frame] = a
    print(f"{frame:8s} {a:9.1%}   ({len(items)})")

print()
if min(acc_all.values()) < 0.8:
    print("*** GATE FAIL: the referent manipulation did not land. ***")
    print("A null Delta would be uninterpretable. Reword the frames before continuing.")
else:
    print("PASS - the model distinguishes self from other. Continue to check 2.")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

frame     accuracy   (n items)
--------------------------------
self          0.0%   (20)
other       100.0%   (20)
fict        100.0%   (20)

*** GATE FAIL: the referent manipulation did not land. ***
A null Delta would be uninterpretable. Reword the frames before continuing.


## Check 2 — reliability ceiling

Split the self-frame data in half, build a distress vector from each half, take the cosine.
Spearman–Brown corrects the half-length estimate to the full instrument.

**Pass condition: ceiling ≥ 0.85.** Below that, a normalized RSI is unstable and you need
more items per cell. The contingency table is in `v2-hostile-review.md` §B6 — a low ceiling
is a scope adjustment, not a dead project.

In [5]:
sys.path.insert(0, WORK)
import importlib, harness_v2 as H
importlib.reload(H)

rng = np.random.default_rng(0)
n_layers = model.config.num_hidden_layers
hs_idx = sorted({max(1, min(n_layers, round(d*n_layers))) for d in [0.25,0.5,0.75,1.0]})
cells = battery["cells"]

acts = {}
for key in ["self_distress","self_neutral","other_distress","other_neutral"]:
    acts[key] = H.extract(model, tok, cells[key], hs_idx, SYS, 8, dev)

print(f"{'depth':>6s} {'ceiling':>9s} {'gate':>7s} {'n needed':>10s}")
print("-" * 36)
ceilings = {}
for i in hs_idx:
    c = H.split_half_ceiling(acts["self_distress"][i], acts["self_neutral"][i], rng)
    sb = c["ceiling_spearman_brown"]; ceilings[i] = sb
    d = round(i/n_layers, 2)
    if sb >= 0.85:
        print(f"{d:6.2f} {sb:9.3f} {'PASS':>7s} {'-':>10s}")
    else:
        need = int(round(30*((0.85/1.15)/max(sb/(2-sb),1e-6))**2))
        print(f"{d:6.2f} {sb:9.3f} {'fail':>7s} {need:>10d}")

best = max(ceilings.values())
print()
print(f"best ceiling across depths: {best:.3f}")
print("PASS - instrument is usable." if best >= 0.85 else
      "GATE FAIL - raise n per cell in build_prompts.py and re-run this notebook.")

 depth   ceiling    gate   n needed
------------------------------------
  0.25     0.954    PASS          -
  0.50     0.954    PASS          -
  0.75     0.952    PASS          -
  1.00     0.949    PASS          -

best ceiling across depths: 0.954
PASS - instrument is usable.


## Check 3 — Δ preview

The primary statistic: within-frame split-half cosine minus cross-frame cosine.
`Δ > 0` with a CI excluding 0 means a self-specific component.

This is a *preview* on one model at one persona. Do not put it in the paper — it is here
so you know whether Friday is worth starting.

In [6]:
print(f"{'depth':>6s} {'cos(s,o)':>9s} {'delta':>8s} {'95% CI':>20s} {'verdict':>10s}")
print("-" * 60)
for i in hs_idx:
    dt = H.delta_test(acts["self_distress"][i], acts["self_neutral"][i],
                      acts["other_distress"][i], acts["other_neutral"][i],
                      rng, n_iter=400)
    cs = H.bootstrap_cross_frame(acts["self_distress"][i], acts["self_neutral"][i],
                                 acts["other_distress"][i], acts["other_neutral"][i],
                                 rng, n_iter=400)
    ci = f"[{dt['ci95'][0]:+.3f},{dt['ci95'][1]:+.3f}]"
    print(f"{i/n_layers:6.2f} {cs['mean']:9.3f} {dt['delta_mean']:+8.3f} {ci:>20s} "
          f"{'SIG' if dt['significant'] else 'ns':>10s}")

print()
print("Read this as a go/no-go on the DESIGN, not as a result.")
print("SIG anywhere -> the full run is worth it.")
print("ns everywhere + ceiling passed -> you likely have a genuine null, which is")
print("   still a paper. Check cos(v_self, v_sentiment) in the full run before")
print("   concluding, in case you found a generic sentiment direction.")

 depth  cos(s,o)    delta               95% CI    verdict
------------------------------------------------------------
  0.25     0.682   +0.233      [+0.159,+0.303]        SIG
  0.50     0.565   +0.349      [+0.290,+0.398]        SIG
  0.75     0.638   +0.267      [+0.202,+0.328]        SIG
  1.00     0.614   +0.294      [+0.220,+0.373]        SIG

Read this as a go/no-go on the DESIGN, not as a result.
SIG anywhere -> the full run is worth it.
ns everywhere + ceiling passed -> you likely have a genuine null, which is
   still a paper. Check cos(v_self, v_sentiment) in the full run before
   concluding, in case you found a generic sentiment direction.


## Gate summary

| check | pass condition | if it fails |
|---|---|---|
| comprehension | self-frame accuracy near ceiling | reword the referent frames — do not proceed |
| ceiling | ≥ 0.85 at some depth | raise n per cell (see contingency table) |
| Δ preview | any depth SIG, **or** a clean null with the ceiling passed | both branches are publishable |

Download `prompts_battery_v2.json` before the session dies.